# Pretraining Objectives

একটি toy tokenized বাক্য নিয়ে চারটি pretraining masking recipe-ই প্রয়োগ করা হয়েছে -- causal LM, masked LM, span corruption ও prefix LM -- সাথে ছাপা হয়েছে প্রতিটি objective মডেলকে ঠিক কোন input খাওয়ায়, প্রতিটি position-এ কী target চায়, আর কোন position-গুলো আসলে loss-এ অবদান রাখে।

"একই architecture, ভিন্ন loss-masking recipe" ধারণাটি সম্পূর্ণ মূর্ত করতে (শুধু মুদ্রিত টেবিল নয়), প্রতিটি objective-এর loss বাস্তবেই গণনা করা হয় `torch.nn.functional.cross_entropy` দিয়ে, এলোমেলো (RANDOM) logits-এর উপর, `ignore_index=-100` ব্যবহার করে (এখানে কোনো প্রশিক্ষিত মডেল নেই -- উদ্দেশ্য masking-এর মেকানিক্স, prediction নয়)। প্রতিটি recipe-র loss mask-এর বাইরের position-গুলোতে loss ঠিক 0.0 আসে -- এর প্রমাণ যে "এখানে loss নেই" মানে সত্যিই সেই position থেকে কোনো gradient signal নেই, মুদ্রিত টেবিলের একটি অব্যবহৃত কলাম নয়।

**চালানোর নিয়ম:** কোষগুলো উপরে থেকে নিচে (Run All) চালান। প্রতিটি অংশের demo নিজের কোষেই চলে; শেষ কোষের `main()` পুরো রানটি একসাথে আরেকবার চালায়।

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

IGNORE_INDEX = -100

## Toy বাক্য ও vocabulary (setup data)

Word-স্তরের token (sub-word/BPE নয়) -- একান্তই যাতে মুদ্রিত টেবিলের প্রতিটি position একটি বাস্তব English শব্দ হিসেবে পড়া যায়।

In [ ]:
# ---------------------------------------------------------------------------
# Toy sentence and vocabulary। Word-level token (sub-word/BPE নয়) -- একান্তই
# যাতে মুদ্রিত টেবিলের প্রতিটি position একটি বাস্তব English শব্দ হিসেবে পড়া যায়।
# ---------------------------------------------------------------------------

SENTENCE = "the cat sat on the mat while the dog slept".split()
#            0    1   2   3   4    5    6    7    8    9

EXTRA_TOKENS = ["[MASK]", "<X>", "<Y>", "<Z>", "<BOS>", "forest"]
vocab = sorted(set(SENTENCE)) + EXTRA_TOKENS
stoi = {tok: i for i, tok in enumerate(vocab)}
VOCAB_SIZE = len(vocab)


def ids(tokens):
    return torch.tensor([stoi[t] for t in tokens], dtype=torch.long)


def print_table(title, rows):
    """rows: (position, input_tok, target_tok_or_dash, loss_yesno, loss_value_or_dash)
    tuple-এর তালিকা।"""
    print(f"\n--- {title} ---")
    print(f"{'pos':>4}{'input':>10}{'predicts (target)':>20}{'in loss?':>14}{'loss value':>12}")
    for pos, inp, tgt, in_loss, loss_val in rows:
        print(f"{pos:>4}{inp:>10}{tgt:>20}{in_loss:>14}{loss_val:>12}")


def compute_losses(logits, target_ids):
    """logits: (T, vocab_size) এলোমেলো 'model output'। target_ids: (T,), যাতে প্রতিটি
    non-loss position-এ IGNORE_INDEX থাকে। প্রতি position-এর loss tensor রিটার্ন করে --
    PyTorch-এর cross_entropy উপেক্ষিত position-এ ঠিক 0.0 রিটার্ন করে।"""
    return F.cross_entropy(logits, target_ids, ignore_index=IGNORE_INDEX, reduction="none")

## 1. Causal LM (Phase 02 Lesson 6)

প্রতিটি position পরের token predict করে, কঠোর causal mask-সহ -- loss প্রতিটি position-এ গণনা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Causal LM (Phase 02 Lesson 6): প্রতিটি position পরের token predict করে,
# কঠোর causal mask, প্রতিটি position-এ loss।
# ---------------------------------------------------------------------------

def causal_lm_demo():
    input_tokens = SENTENCE[:-1]          # token 0..8
    target_tokens = SENTENCE[1:]          # token 1..9 (এক ঘর শিফট করা)
    T = len(input_tokens)

    target_ids = ids(target_tokens)       # প্রতিটি position-ই supervised
    logits = torch.randn(T, VOCAB_SIZE)
    losses = compute_losses(logits, target_ids)

    rows = [
        (i, input_tokens[i], target_tokens[i], "yes", f"{losses[i].item():.3f}")
        for i in range(T)
    ]
    print_table("1. CAUSAL LM  (decoder-only, e.g. GPT)", rows)
    print(f"    -> attention: causal mask (each position sees only positions <= itself)")
    print(f"    -> loss positions: {T}/{T}  (every position supervised)")
    return losses


# এই অংশের demo।
causal_losses = causal_lm_demo()

## 2. Masked LM (BERT)

কিছু position corrupt হয়; bidirectional context থেকে তাদের ORIGINAL পরিচয় predict করতে বলা হয়; loss শুধু সেই corrupted position-গুলোতে। এখানে 3টি masked position (BERT-এর প্রকৃত ~15%-এর চেয়ে বেশি) -- একান্তই যাতে 10-শব্দের toy বাক্য BERT-এর তিনটি masking উপ-ক্ষেত্র-ই দৃশ্যমানভাবে দেখাতে পারে: `[MASK]` প্রতিস্থাপন, এলোমেলো token প্রতিস্থাপন এবং অপরিবর্তিত রাখা।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Masked LM (BERT): কিছু position corrupt করুন, bidirectional context থেকে
# তাদের ORIGINAL পরিচয় predict করুন, loss শুধু corrupted position-গুলোতে।
# এখানে 3টি masked position (BERT-এর প্রকৃত ~15%-এর চেয়ে বেশি) একান্তই যাতে
# 10-শব্দের toy বাক্য BERT-এর তিনটি masking উপ-ক্ষেত্র দৃশ্যমানভাবে দেখাতে পারে:
# [MASK] প্রতিস্থাপন, random-token প্রতিস্থাপন এবং অপরিবর্তিত।
# ---------------------------------------------------------------------------

def masked_lm_demo():
    input_tokens = list(SENTENCE)         # সম্পূর্ণ 10-token sequence, bidirectional
    target_tokens = [None] * len(input_tokens)

    masked_positions = {
        2: "[MASK]",   # 80%-ক্ষেত্র: [MASK] দিয়ে প্রতিস্থাপন, মূল "sat" predict করুন
        5: SENTENCE[5],  # 10%-ক্ষেত্র: অপরিবর্তিত রাখা ("mat"), তবুও loss position
        8: "forest",    # 10%-ক্ষেত্র: এলোমেলো অন্য token দিয়ে প্রতিস্থাপন, মূল "dog" predict করুন
    }
    for pos, replacement in masked_positions.items():
        target_tokens[pos] = SENTENCE[pos]   # target = ORIGINAL token, সবসময়
        input_tokens[pos] = replacement       # input = (সম্ভবত corrupt করা) token

    T = len(input_tokens)
    target_ids = torch.tensor(
        [stoi[target_tokens[i]] if target_tokens[i] is not None else IGNORE_INDEX for i in range(T)]
    )
    logits = torch.randn(T, VOCAB_SIZE)
    losses = compute_losses(logits, target_ids)

    rows = [
        (
            i, input_tokens[i],
            target_tokens[i] if target_tokens[i] is not None else "-",
            "yes" if i in masked_positions else "no",
            f"{losses[i].item():.3f}",
        )
        for i in range(T)
    ]
    print_table("2. MASKED LM  (encoder-only, e.g. BERT)", rows)
    print(f"    -> attention: fully bidirectional (no mask at all)")
    print(f"    -> loss positions: {len(masked_positions)}/{T}  (only the corrupted positions)")
    return losses


# এই অংশের demo।
mlm_losses = masked_lm_demo()

## 3. Span corruption (T5)

Encoder দিক থেকে ধারাবাহিক spans-কে sentinel দিয়ে প্রতিস্থাপন করা হয় (সেখানে loss নেই); decoder শুধু অনুপস্থিত বিষয়বস্তু generate করে, sentinel দিয়ে ট্যাগ করা, একটি ছোট target sequence হিসেবে -- সেই (ছোট) target-এর প্রতিটি position-এ loss।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Span corruption (T5): encoder দিক থেকে CONTIGUOUS spans-কে sentinel দিয়ে
# প্রতিস্থাপন করুন (সেখানে loss নেই); decoder শুধু অনুপস্থিত বিষয়বস্তু generate
# করে, sentinel দিয়ে ট্যাগ করা, একটি ছোট target sequence হিসেবে, সাথে সেই
# (ছোট) target-এর প্রতিটি position-এ loss।
# ---------------------------------------------------------------------------

def span_corruption_demo():
    # Encoder দিক: দুটি contiguous span সরিয়ে sentinel দিয়ে প্রতিস্থাপিত।
    # span A = position 1-2 ("cat sat") -> <X>; span B = position 7-8 ("the dog") -> <Y>
    encoder_input = ["the", "<X>", "on", "the", "mat", "while", "<Y>", "slept"]

    # Decoder দিক: teacher-forced input শুরু হয় <BOS> দিয়ে; target শুধু
    # অনুপস্থিত spans-গুলোকেই পুনর্গঠন করে, প্রতিটি যার প্রতিস্থাপনকারী
    # sentinel দিয়ে ট্যাগ করা, শেষে চূড়ান্ত sentinel <Z> -- "আর কোনো span নেই"।
    decoder_input = ["<BOS>", "<X>", "cat", "sat", "<Y>", "the", "dog"]
    decoder_target = ["<X>", "cat", "sat", "<Y>", "the", "dog", "<Z>"]
    T = len(decoder_input)

    target_ids = ids(decoder_target)   # প্রতিটি decoder position-ই supervised
    logits = torch.randn(T, VOCAB_SIZE)
    losses = compute_losses(logits, target_ids)

    print(f"\n--- 3. SPAN CORRUPTION  (encoder-decoder, e.g. T5) ---")
    print(f"    encoder input (bidirectional, NO loss): {' '.join(encoder_input)}")
    rows = [
        (i, decoder_input[i], decoder_target[i], "yes", f"{losses[i].item():.3f}")
        for i in range(T)
    ]
    print_table("decoder side (causal, cross-attends to encoder)", rows)
    print(f"    -> loss positions: {T}/{T} decoder positions "
          f"(but target is only {T} tokens vs. the original 10-token sentence --")
    print(f"       reconstructing just the missing spans is cheaper than reconstructing everything)")
    return losses


# এই অংশের demo।
span_losses = span_corruption_demo()

## 4. Prefix LM (UniLM / PaLM-ধাঁচ)

Prefix-এর উপর bidirectional, continuation-এর উপর causal attention; loss শুধু continuation position-গুলোতে।

In [ ]:
# ---------------------------------------------------------------------------
# 4. Prefix LM (UniLM / PaLM-ধাঁচ): PREFIX-এর উপর bidirectional, continuation-এর
# উপর causal। Loss শুধু continuation position-গুলোতে।
# ---------------------------------------------------------------------------

def prefix_lm_demo():
    k = 4   # prefix দৈর্ঘ্য: "the cat sat on" bidirectional attention পায়, loss নেই
    input_tokens = SENTENCE[:-1]     # causal LM-এর মতোই আকৃতি: token 0..8
    target_tokens = SENTENCE[1:]     # token 1..9
    T = len(input_tokens)

    target_ids = torch.tensor(
        [stoi[target_tokens[i]] if i >= k else IGNORE_INDEX for i in range(T)]
    )
    logits = torch.randn(T, VOCAB_SIZE)
    losses = compute_losses(logits, target_ids)

    rows = [
        (
            i, input_tokens[i], target_tokens[i] if i >= k else "-",
            "no (prefix)" if i < k else "yes", f"{losses[i].item():.3f}",
        )
        for i in range(T)
    ]
    print_table("4. PREFIX LM  (hybrid, e.g. UniLM / PaLM variant)", rows)
    print(f"    -> attention: positions 0..{k - 1} (prefix) see each other bidirectionally;")
    print(f"       positions {k}..{T - 1} attend causally (to the whole prefix + earlier continuation)")
    print(f"    -> loss positions: {T - k}/{T}  (continuation only; prefix contributes no loss)")
    return losses


def main():
    print("=" * 78)
    print("SAME SENTENCE, FOUR PRETRAINING OBJECTIVES")
    print("=" * 78)
    print(f"Sentence: {' '.join(SENTENCE)}")
    print(f"Vocabulary ({VOCAB_SIZE} tokens): {vocab}")

    causal_losses = causal_lm_demo()
    mlm_losses = masked_lm_demo()
    span_losses = span_corruption_demo()
    prefix_losses = prefix_lm_demo()

    print("\n" + "=" * 78)
    print("SUMMARY: LOSS POSITIONS PER OBJECTIVE (same underlying Transformer math)")
    print("=" * 78)
    print(f"{'objective':22s}{'total positions':>18}{'loss positions':>16}{'fraction':>12}")
    for name, losses in [
        ("Causal LM", causal_losses), ("Masked LM", mlm_losses),
        ("Span corruption", span_losses), ("Prefix LM", prefix_losses),
    ]:
        total = losses.numel()
        active = (losses != 0.0).sum().item()
        print(f"{name:22s}{total:>18}{active:>16}{active / total:>12.0%}")

    print("\n-> Every objective above ran through the exact same mechanism: random")
    print("   logits, torch.nn.functional.cross_entropy with ignore_index=-100.")
    print("   The ONLY thing that changed between objectives was which target")
    print("   value (real token id, or -100) sat at each position -- confirmed")
    print("   numerically above, since every ignored position's loss came back")
    print("   as exactly 0.000. Causal LM and span corruption supervise every")
    print("   position of their (different-length) sequences; masked LM and")
    print("   prefix LM each supervise only a subset. Swapping which subset is")
    print("   supervised, and whether the attention mask is causal or bidirectional,")
    print("   IS the difference between a GPT-style, BERT-style, T5-style, and")
    print("   UniLM/PaLM-style pretraining run -- not a different Transformer.")


# এই অংশের demo।
prefix_losses = prefix_lm_demo()

In [ ]:
main()